# 日本株の収益性ファクター分析:期待収益性 vs 実績収益性

参考論文:Wahal (2025) "Expected Profitability"

**このノートブックの流れ**

| 章 | 内容 |
|---|---|
| 0 | 設定、DB 接続、データ取得関数、データ読み込み |
| 1 | 財務データの整形(欠損値・会計基準・前期との対応) |
| 2 | 収益性指標の計算(ROA, ROE, GP/A, OP, CBOP, CP) |
| 3 | 組成年への割り当てと会計基準の選択、市場データとの結合 |
| 4 | 第1段階:将来の収益性の予測(期待収益性の推定)と予測精度 |
| 5 | 第2段階:Fama-MacBeth 回帰 |
| 6 | 2×3 ポートフォリオによるファクターリターン |
| 7 | 業種別分析(規模調整あり) |
| 8 | まとめ:指標の比較と期待収益性の効果 |
| 9 | 結果の保存 |

分析の設計と判断の根拠は `analysis_notes.md` にまとめています。

## 0. 設定

分析の前提はここで変更できます。

- `CTYPE_PRIORITY`:同じ決算期に複数の会計基準のデータがある場合の優先順位
  (1: SEC式 → 3: IFRS → 2: 日本式)
- `ACCOUNTING_UNIT`:財務データの単位(百万円と仮定)。時価総額は円なので、B/M の計算で単位をそろえます。
- `FIRST_STAGE_YEARS`:期待収益性の係数として、直近何年分の年次回帰の係数を平均するか(1 なら論文と同じく単年)

In [ ]:
import calendar
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

try:
    display
except NameError:  # Jupyter 以外で実行したとき
    display = print

# ===== 設定 =====
MISSING_SENTINEL = -999999998        # IDSQE 系の欠損値(これ以下の値はすべて欠損扱い)
CTYPE_PRIORITY = ["1", "3", "2"]     # 会計基準の優先順位(1: SEC, 3: IFRS, 2: 日本)
CTYPE_NAME = {"1": "SEC", "2": "日本", "3": "IFRS"}
ACCOUNTING_UNIT = 1e6                # 財務データの単位(百万円)→ 円に直す倍率

START_YEAR = 1990                    # 最初のポートフォリオ組成年
END_YEAR = 2024                      # 最後のポートフォリオ組成年
FORMATION_MONTH = 6                  # 毎年この月の月末に組成し、翌月から12か月保有
AVAIL_FALLBACK_MONTHS = 2            # 発表日・短信入手日が欠損のとき、決算期末 + 2か月で入手可能とみなす
MAX_STALENESS_MONTHS = 15            # 組成時点から見て、決算期末がこれより古い決算は使わない

WINSOR = (0.01, 0.99)                # 毎年のクロスセクションで winsorize する分位点
FIRST_STAGE_YEARS = 1                # 期待収益性の係数:直近何年分の係数を平均するか
MICRO_PCT = 0.20                     # マイクロキャップの閾値(TOPIX 構成銘柄の時価総額の 20 パーセンタイル)
MIN_OBS_CS = 30                      # クロスセクション回帰に必要な最小銘柄数
NW_LAGS_MONTHLY = 12                 # 月次系列の Newey-West ラグ
NW_LAGS_ANNUAL = 3                   # 年次系列の Newey-West ラグ

MEASURES = ["ROA", "ROE", "GPA", "OP", "CBOP", "CP"]
# 分母:asset = 前期末総資産、equity = 前期末自己資本
SCALE = {"ROA": "asset", "GPA": "asset", "CBOP": "asset",
         "ROE": "equity", "OP": "equity", "CP": "equity"}

# 除外する東証33業種(電気・ガス、銀行、証券・商品先物、保険、その他金融)
EXCLUDE_IND33 = {"4050", "7050", "7100", "7150", "7200"}

CACHE_PATH = Path("cache_raw_data.pkl")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
FF5_PATH = Path("data/Japan_5_Factors.csv")   # Ken French サイトの日本の5ファクター(月次)

## 0-1. DB 接続

`temp1.ipynb` で使っている DB 接続用の関数 `fetch_data` と `get_data` を、このセルで import するか定義してください。
キャッシュ(`cache_raw_data.pkl`)がある場合は DB に接続しないので、未定義でも実行できます。

In [ ]:
# 例:from db_utils import fetch_data, get_data
# ↑ temp1.ipynb と同じ import 文・定義をここに書く

if "fetch_data" not in globals() or "get_data" not in globals():
    print("[注意] fetch_data / get_data が未定義です。キャッシュがない場合、データ取得でエラーになります。")

## 0-2. データ取得関数

`temp1.ipynb` の取得関数をそのまま載せています。変更点は1つだけです。

- `get_topix_data`:SELECT の `[ITEM]` と `[ITEM_DATA]` の間にカンマが抜けていたのを修正

業種の `get_industry` は、業種名テーブルとの JOIN 条件に業種コードが入っておらず `IND_NAME` が正しくない
可能性がありますが、分析では業種コード(`ITEM_DATA`)だけを使うので、関数はそのままにしています。
欠損値(-999999998)の置き換えは、分析側(1章)で行います。

In [ ]:
# PL項目の取得
def get_pl_data():
    PL_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FTERM]        -- 決算期 (int yyyymm)
        ,[CTYPE]        -- 会計基準 (1: SEC 2: 日本式 3: IFRS)
        ,[GPLS]         -- 売上総利益
        ,[OPINCM]       -- 営業利益
        ,[CHAGE]        -- 支払利息
        ,[PLCURREN]     -- 為替換算調整勘定，のちのちの分析で必要かもなので一応
        ,[NETINCM]      -- 当期純利益
        ,[RD]           -- 研究開発費
    FROM [IDSQE].[dbo].[NRIFIN_CON_PL4]
    """
    df = fetch_data(PL_QUERY, 'idsqe')
    # 欠損値は-999999998
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# BS項目の取得
def get_bs_data():
    BS_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FTERM]        -- 決算期 (int yyyymm)
        ,[CTYPE]        -- 会計基準 (1: SEC 2: 日本式 3: IFRS)
        ,[CASH]         -- 現金・預金，のちのちの分析で必要かもなので一応
        ,[ACRC]         -- 売上債権，アクルーアルの計算要
        ,[INVENT]       -- 棚卸資産
        ,[INVSCRT]      -- 投資有価証券，別の分析でのちほど用いる
        ,[TOTALBS]      -- 資産合計
        ,[ACPY]         -- 買入債務
        ,[CAPSTTL]      -- 株主資本合計
        ,[NTASTTL]      -- 純資産合計，株主資本合計が欠損しているときに用いる
        ,[EQTY]         -- 資本合計，株主資本合計が欠損しているとき
    FROM [IDSQE].[dbo].[NRIFIN_CON_BS4]
    """
    df = fetch_data(BS_QUERY, 'idsqe')
    # 欠損値は-999999998
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# 決算詳細
def get_accounting_details():
    DETAILS_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FTERM]            -- 決算期 int yyyymm
        ,[CTYPE]            -- 会計基準 1: SEC式 2: 日本式 3: IFRS式
        ,[SETTLEMENT_FLG]   -- 本中区分 1: 本決算 2: 中間決算 3: 第1四半期決算 4: 第3四半期決算
        ,[FDATE]            -- 決算年月日 char yyyymmdd
        ,[NMONTH]           -- 決算月数 通常12
        ,[ANNOUNCE]         -- 決算発表日，[RCVD]より欠損(-999999998)が多い
        ,[RCVD]             -- 決算短信入手日
    FROM [IDSQE].[dbo].[NRIFIN_CON_AT4]
    """
    df = fetch_data(DETAILS_QUERY, 'idsqe')
    # 欠損値は-999999998
    # 決算発表、短信入手日どちらも欠損している場合は、決算期プラス2か月する（日本の法律）
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# 会社属性
# NRI_CODE -> 証券コード
def get_nri_to_ticker_table():
    TABLE_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FROM_DATE]        -- int yyyymmdd
        ,[TO_DATE]          -- int yyyymmdd
        ,[ITEM_DATA]        -- 証券コード
    FROM [IDSQE].[dbo].[COMPANY_CODE]
    WHERE ITEM = 'OLDCD'            -- 証券コード
    """
    df = fetch_data(TABLE_QUERY, 'idsqe')
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# 業種コード
def get_industry():
    IND_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FROM_DATE]        -- int yyyymmdd
        ,[TO_DATE]          -- int yyyymmdd
        ,c.[ITEM_DATA]
        ,n.[IND_NAME]
    FROM [IDSQE].[dbo].[COMPANY_INDUSTRY_CLASS] c
    JOIN [IDSQE].[dbo].[INDUSTRY_CLASS_NAME] n ON c.ITEM = n.IND
    WHERE [ITEM] = 'IND33'   -- 東証33業種
    """
    df = fetch_data(IND_QUERY, 'idsqe')
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# TOPIX構成銘柄
def get_topix_data():
    query = """
    SELECT
        [NRI_CODE]
        ,[FROM_DATE]
        ,[TO_DATE]
        ,[ITEM]
        ,[ITEM_DATA]
    FROM [IDSQE].[dbo].[STOCK_ISSUE_INDEX]
    WHERE [ITEM] = 'TOPIX';
    """
    return get_data(query, "idsqe")


# 時価総額の取得
def get_market_cap_data():
    # 月次時価総額と、各月末時点で有効な証券コードを取得
    query = """
    SELECT
        prc.[DATE] AS [date],
        prc.[CAP] AS market_cap,
        SUBSTRING(RTRIM(id.[ID]), 3, LEN(RTRIM(id.[ID])))
            AS security_code
    FROM [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_M_PRC] AS prc
    CROSS APPLY (
        -- YYYYMMから月末日（YYYYMMDD）を作成
        SELECT CONVERT(
            int,
            CONVERT(
                char(8),
                EOMONTH(DATEFROMPARTS(
                    prc.[DATE] / 100,
                    prc.[DATE] % 100,
                    1
                )),
                112
            )
        ) AS month_end
    ) AS d
    INNER JOIN [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_ID] AS id
        ON prc.[BID] = id.[BID]
        AND id.[IDTYPE] = 'LOCALID'
        AND id.[FDATE] <= d.[month_end]
        AND (id.[TDATE] >= d.[month_end] OR id.[TDATE] IS NULL);
    """
    df = get_data(query, "riskmodels")

    # 証券コードと数値列を整形
    df["security_code"] = df["security_code"].astype("string").str.strip().str.upper()
    df["date"] = pd.to_numeric(df["date"], errors="raise").astype(int)
    df["market_cap"] = (
        pd.to_numeric(df["market_cap"], errors="raise")
        .astype(float)
        .replace([np.inf, -np.inf], np.nan)
    )
    # 月次Period列を追加
    df["month"] = pd.to_datetime(df["date"].astype(str), format="%Y%m").dt.to_period("M")
    return df


# 銘柄リターンの取得
def get_return_data():
    # 月次リターンと、各月初時点で有効な証券コードを取得
    query = """
    SELECT
        rtn.[DATE],
        SUBSTRING(RTRIM(id.[ID]), 3, LEN(RTRIM(id.[ID])))
            AS security_code,
        rtn.[RTN]
    FROM [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_M_RTN] AS rtn
    INNER JOIN [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_ID] AS id
        ON rtn.[BID] = id.[BID]
    WHERE id.[IDTYPE] = 'LOCALID'
      AND TRY_CONVERT(date, CONCAT(rtn.[DATE], '01'), 112)
          >= TRY_CONVERT(date, CONVERT(varchar(8), id.[FDATE]), 112)
      AND (
          id.[TDATE] IS NULL
          OR TRY_CONVERT(date, CONCAT(rtn.[DATE], '01'), 112)
             <= TRY_CONVERT(date, CONVERT(varchar(8), id.[TDATE]), 112)
      );
    """
    df = get_data(query, "riskmodels")

    # 証券コード・年月を整形し、リターンを%から小数へ変換
    df["security_code"] = df["security_code"].astype("string").str.strip().str.upper()
    dates = pd.to_numeric(df["DATE"], errors="raise").astype(int)
    df["month"] = pd.to_datetime(dates.astype(str), format="%Y%m").dt.to_period("M")
    df["ret"] = pd.to_numeric(df["RTN"], errors="raise").astype(float) / 100
    return df[["security_code", "month", "ret"]].copy()

### 東証33業種コード → 業種名・TOPIX-17業種

取得した業種テーブルは、業種名テーブルとの JOIN 条件が業種コードを含んでいないため、
`IND_NAME` が正しく対応していない可能性があります。ここでは業種コード(`ITEM_DATA`)だけを使い、
名称と17業種への対応は東証の標準コード表から付けます。

In [ ]:
IND33_NAME = {
    "0050": "水産・農林業", "1050": "鉱業", "2050": "建設業", "3050": "食料品", "3100": "繊維製品",
    "3150": "パルプ・紙", "3200": "化学", "3250": "医薬品", "3300": "石油・石炭製品", "3350": "ゴム製品",
    "3400": "ガラス・土石製品", "3450": "鉄鋼", "3500": "非鉄金属", "3550": "金属製品", "3600": "機械",
    "3650": "電気機器", "3700": "輸送用機器", "3750": "精密機器", "3800": "その他製品", "4050": "電気・ガス業",
    "5050": "陸運業", "5100": "海運業", "5150": "空運業", "5200": "倉庫・運輸関連業", "5250": "情報・通信業",
    "6050": "卸売業", "6100": "小売業", "7050": "銀行業", "7100": "証券、商品先物取引業", "7150": "保険業",
    "7200": "その他金融業", "8050": "不動産業", "9050": "サービス業",
}
IND33_TO_17 = {
    "0050": "食品", "3050": "食品",
    "1050": "エネルギー資源", "3300": "エネルギー資源",
    "2050": "建設・資材", "3400": "建設・資材", "3550": "建設・資材",
    "3100": "素材・化学", "3150": "素材・化学", "3200": "素材・化学",
    "3250": "医薬品",
    "3350": "自動車・輸送機", "3700": "自動車・輸送機",
    "3450": "鉄鋼・非鉄", "3500": "鉄鋼・非鉄",
    "3600": "機械",
    "3650": "電機・精密", "3750": "電機・精密",
    "3800": "情報通信・サービスその他", "5250": "情報通信・サービスその他", "9050": "情報通信・サービスその他",
    "4050": "電力・ガス",
    "5050": "運輸・物流", "5100": "運輸・物流", "5150": "運輸・物流", "5200": "運輸・物流",
    "6050": "商社・卸売", "6100": "小売", "7050": "銀行",
    "7100": "金融(除く銀行)", "7150": "金融(除く銀行)", "7200": "金融(除く銀行)",
    "8050": "不動産",
}

### 共通の関数

In [ ]:
def clean_idsqe(df):
    """IDSQE 系テーブルの整形:欠損値(-999999998)を NaN に、文字列の前後空白を除去"""
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c]):
            df[c] = df[c].mask(df[c] <= MISSING_SENTINEL)
        elif pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip()
    return df


def norm_code(s):
    """'2'・2・2.0 のどれで来ても '2' という文字列にそろえる(会計基準・本中区分用)"""
    num = pd.to_numeric(s, errors="coerce")
    out = s.astype(str).str.strip()
    return out.where(num.isna(), num.round().astype("Int64").astype(str))


def norm_sec(s):
    """銘柄コード・証券コードを文字列にそろえる"""
    return s.astype(str).str.strip().str.upper()


def to_date(s):
    """yyyymmdd の整数 → Timestamp(0・99999999・欠損は NaT)"""
    v = pd.to_numeric(s, errors="coerce")
    v = v.where((v > 19000000) & (v < 21000000))
    return pd.to_datetime(v.astype("Int64").astype(str), format="%Y%m%d", errors="coerce")


def to_midx(s):
    """月(Period・'YYYY-MM'・yyyymm 整数)→ 通し月番号 year*12 + month - 1"""
    if pd.api.types.is_numeric_dtype(s):
        v = s.astype(int)
        return (v // 100) * 12 + v % 100 - 1
    dt = pd.to_datetime(s.astype(str))
    return dt.dt.year * 12 + dt.dt.month - 1


def midx_to_ts(m):
    m = np.asarray(m, dtype=int)
    return pd.to_datetime({"year": m // 12, "month": m % 12 + 1, "day": 1}) + pd.offsets.MonthEnd(0)


def formation_date(y):
    """組成日(例:y 年 6 月末)"""
    return pd.Timestamp(int(y), FORMATION_MONTH, calendar.monthrange(int(y), FORMATION_MONTH)[1])


def formation_int(y):
    d = formation_date(y)
    return d.year * 10000 + d.month * 100 + d.day


def winsorize_by(df, cols, by, limits=WINSOR):
    """by ごと(毎年)に cols を winsorize"""
    df = df.copy()
    for c in cols:
        g = df.groupby(by)[c]
        lo = g.transform(lambda x: x.quantile(limits[0]))
        hi = g.transform(lambda x: x.quantile(limits[1]))
        df[c] = df[c].clip(lo, hi)
    return df


def ols_nw(y, X, lags):
    """OLS(定数項あり)と Newey-West の t 値。X は (T, k) 。k=0 なら平均の検定"""
    y = np.asarray(y, dtype=float)
    X = np.asarray(X, dtype=float).reshape(len(y), -1)
    X = np.column_stack([np.ones(len(y)), X])
    ok = ~np.isnan(y) & ~np.isnan(X).any(axis=1)
    y, X = y[ok], X[ok]
    T = len(y)
    if T <= X.shape[1] + 1:
        return np.full(X.shape[1], np.nan), np.full(X.shape[1], np.nan), T
    XtX_inv = np.linalg.pinv(X.T @ X)
    b = XtX_inv @ X.T @ y
    u = y - X @ b
    Xu = X * u[:, None]
    S = Xu.T @ Xu
    for l in range(1, min(lags, T - 1) + 1):
        w = 1 - l / (lags + 1)
        G = Xu[l:].T @ Xu[:-l]
        S += w * (G + G.T)
    V = XtX_inv @ S @ XtX_inv
    se = np.sqrt(np.diag(V))
    return b, b / se, T


def mean_t(x, lags=NW_LAGS_MONTHLY):
    """時系列平均と Newey-West の t 値"""
    x = pd.Series(x).dropna()
    b, t, T = ols_nw(x.values, np.empty((len(x), 0)), lags)
    return b[0], t[0], T


def ols_cs(y, X):
    """クロスセクション OLS:係数・自由度調整済み R²・N"""
    Xc = np.column_stack([np.ones(len(y)), X])
    b, *_ = np.linalg.lstsq(Xc, y, rcond=None)
    e = y - Xc @ b
    n, k = Xc.shape
    r2 = 1 - (e @ e) / ((y - y.mean()) @ (y - y.mean()))
    adj = 1 - (1 - r2) * (n - 1) / (n - k)
    return b, adj, n


def fama_macbeth(df, ycol, xcols, wcol=None, tcol="midx", min_obs=MIN_OBS_CS):
    """毎期のクロスセクション回帰(wcol があれば加重最小二乗)。
    戻り値:coef(期×[const]+xcols)、n、adjr2、iqr(各説明変数の毎期の四分位範囲)"""
    cols = [tcol, ycol] + xcols + ([wcol] if wcol else [])
    d = df[cols].dropna().sort_values(tcol)
    if wcol:
        d = d[d[wcol] > 0]
    t = d[tcol].to_numpy()
    Y = d[ycol].to_numpy(float)
    X = d[xcols].to_numpy(float)
    W = d[wcol].to_numpy(float) if wcol else None
    cuts = np.flatnonzero(np.diff(t)) + 1
    starts = np.r_[0, cuts]
    ends = np.r_[cuts, len(t)]
    rows, iqr_rows = {}, {}
    for s, e in zip(starts, ends):
        n = e - s
        if n < max(min_obs, len(xcols) + 5):
            continue
        y_, X_ = Y[s:e], X[s:e]
        Xc = np.column_stack([np.ones(n), X_])
        if W is not None:
            w = W[s:e] / W[s:e].sum()
            sw = np.sqrt(w)
            b, *_ = np.linalg.lstsq(Xc * sw[:, None], y_ * sw, rcond=None)
            res = y_ - Xc @ b
            ybar = (w * y_).sum()
            r2 = 1 - (w * res ** 2).sum() / (w * (y_ - ybar) ** 2).sum()
        else:
            b, *_ = np.linalg.lstsq(Xc, y_, rcond=None)
            res = y_ - Xc @ b
            r2 = 1 - (res @ res) / ((y_ - y_.mean()) @ (y_ - y_.mean()))
        k = Xc.shape[1]
        rows[t[s]] = np.r_[b, n, 1 - (1 - r2) * (n - 1) / (n - k)]
        q75, q25 = np.percentile(X_, [75, 25], axis=0)
        iqr_rows[t[s]] = q75 - q25
    res = pd.DataFrame.from_dict(rows, orient="index", columns=["const"] + xcols + ["n", "adjr2"]).sort_index()
    iqr = pd.DataFrame.from_dict(iqr_rows, orient="index", columns=xcols).sort_index()
    return {"coef": res[["const"] + xcols], "n": res["n"], "adjr2": res["adjr2"], "iqr": iqr}


def summarize_fm(res, lags=NW_LAGS_MONTHLY, scale=100):
    """FM 回帰の結果:平均係数(×100)と Newey-West t 値"""
    out = {}
    for c in res["coef"].columns:
        m, t, _ = mean_t(res["coef"][c], lags)
        out[c] = {"係数": m * scale, "t値": t}
    out = pd.DataFrame(out).T
    out.loc["平均N", "係数"] = res["n"].mean()
    out.loc["平均adjR2", "係数"] = res["adjr2"].mean()
    return out


def asof_lookup(keys, table, value_cols, date_col):
    """NRI_CODE と日付(yyyymmdd 整数)で、FROM_DATE ≤ 日付 ≤ TO_DATE の行を引く"""
    k = keys[["NRI_CODE", date_col]].drop_duplicates()
    m = k.merge(table, on="NRI_CODE", how="inner")
    m = m[(m["FROM_DATE"] <= m[date_col]) & (m[date_col] <= m["TO_DATE"])]
    m = m.sort_values("FROM_DATE").drop_duplicates(["NRI_CODE", date_col], keep="last")
    return m[["NRI_CODE", date_col] + value_cols]

## 0-3. データ読み込み

上で定義した取得関数で DB からデータを取得し、キャッシュ(`cache_raw_data.pkl`)に保存します。

- 2回目以降はキャッシュを読むので、DB には接続しません。
- DB から取り直したいときは `USE_CACHE = False` にしてください。

In [ ]:
USE_CACHE = True

def load_raw_data(use_cache=USE_CACHE):
    if "取得データ" in globals():  # temp1.ipynb の取得結果がカーネルに残っている場合
        print("既存の `取得データ` を使います")
        return globals()["取得データ"]
    if use_cache and CACHE_PATH.exists():
        print(f"キャッシュ {CACHE_PATH} を読み込みます")
        with open(CACHE_PATH, "rb") as f:
            return pickle.load(f)
    funcs = {
        "PL": get_pl_data, "BS": get_bs_data, "決算詳細": get_accounting_details,
        "証券コード対応表": get_nri_to_ticker_table, "業種": get_industry,
        "TOPIX構成銘柄": get_topix_data, "時価総額": get_market_cap_data, "月次リターン": get_return_data,
    }
    data = {}
    for name, fn in funcs.items():
        print(f"取得中:{name}")
        data[name] = fn()
    with open(CACHE_PATH, "wb") as f:
        pickle.dump(data, f)
    return data

raw = load_raw_data()
for k, v in raw.items():
    print(f"{k}: {v.shape}")

## 1. 財務データの整形

1. 欠損値 `-999999998` を NaN に置き換える。
2. 決算詳細から **本決算(本中区分 = 1)かつ決算月数 = 12** の行だけを残す(中間・四半期・変則決算を除く)。
3. PL・BS・決算詳細を (NRI_CODE, 決算期, 会計基準) で結合する。会計基準ごとに別々の行のまま持ちます。
4. 入手可能日 = 決算発表日 → 決算短信入手日 → 決算期末 + 2か月 の順で決める。

In [ ]:
KEY = ["NRI_CODE", "FTERM", "CTYPE"]

pl = clean_idsqe(raw["PL"])
bs = clean_idsqe(raw["BS"])
at = clean_idsqe(raw["決算詳細"])
for d in (pl, bs, at):
    d["NRI_CODE"] = norm_sec(d["NRI_CODE"])
    d["CTYPE"] = norm_code(d["CTYPE"])
    d["FTERM"] = pd.to_numeric(d["FTERM"]).astype(int)
at["SETTLEMENT_FLG"] = norm_code(at["SETTLEMENT_FLG"])

print("会計基準ごとの行数(決算詳細・全区分)")
display(at["CTYPE"].map(CTYPE_NAME).fillna(at["CTYPE"]).value_counts())

at = at[(at["SETTLEMENT_FLG"] == "1") & (at["NMONTH"] == 12)]


def dedupe(df, name):
    n = df.duplicated(KEY).sum()
    if n:
        print(f"[注意] {name}: (NRI_CODE, 決算期, 会計基準) の重複が {n} 行 → 先頭の行を残します")
        df = df.drop_duplicates(KEY)
    return df


at, pl, bs = dedupe(at, "決算詳細"), dedupe(pl, "PL"), dedupe(bs, "BS")
pl = pl.drop(columns=[c for c in pl.columns if c in ("SETTLEMENT_FLG",)], errors="ignore")
acct = at.merge(pl, on=KEY, how="inner").merge(bs, on=KEY, how="inner")
print(f"本決算・12か月の財務データ:{len(acct):,} 行")

# 決算期の通し月番号、決算期末日、入手可能日
acct["fy_idx"] = (acct["FTERM"] // 100) * 12 + acct["FTERM"] % 100 - 1
fterm_end = midx_to_ts(acct["fy_idx"].to_numpy())
acct["fdate"] = to_date(acct["FDATE"]).fillna(pd.Series(fterm_end.values, index=acct.index))
ann = to_date(acct["ANNOUNCE"])
rcv = to_date(acct["RCVD"])
fallback = acct["fdate"] + pd.offsets.MonthEnd(AVAIL_FALLBACK_MONTHS)
acct["avail_date"] = ann.where(ann >= acct["fdate"]).fillna(rcv.where(rcv >= acct["fdate"])).fillna(fallback)
acct["avail_src"] = np.select([ann.notna() & (ann >= acct["fdate"]), rcv.notna() & (rcv >= acct["fdate"])],
                              ["決算発表日", "短信入手日"], default="期末+2か月")
display(acct["avail_src"].value_counts())

# 欠損率(会計基準別)
check_cols = ["GPLS", "OPINCM", "CHAGE", "NETINCM", "RD", "ACRC", "INVENT", "ACPY",
              "TOTALBS", "CAPSTTL", "EQTY", "NTASTTL"]
print("項目ごとの欠損率(会計基準別)")
display(acct.groupby(acct["CTYPE"].map(CTYPE_NAME).fillna(acct["CTYPE"]))[check_cols]
        .agg(lambda x: x.isna().mean()).T)

## 2. 収益性指標の計算

**前期の値は、同じ会計基準の行からだけ取ります。** 当期と前期(決算期 −12か月)を
(NRI_CODE, 会計基準, 決算期) で結合するので、分子(当期)と分母(前期末)の会計基準は必ず一致します。

| 指標 | 分子 | 分母 |
|---|---|---|
| ROA | 営業利益 | 総資産(t−1) |
| ROE | 当期純利益 | 自己資本(t−1) |
| GPA | 売上総利益 | 総資産(t−1) |
| OP | 営業利益 − 支払利息 | 自己資本(t−1) |
| CBOP | 営業利益 + 研究開発費 − アクルーアル | 総資産(t−1) |
| CP | 営業利益 − 支払利息 − アクルーアル | 自己資本(t−1) |

- 自己資本 = 株主資本合計 → 資本合計 → 純資産合計 の順で、最初に値があるもの(0 以下は欠損扱い)
- アクルーアル = Δ売上債権 + Δ棚卸資産 − Δ買入債務
  - 当期・前期とも欠損の項目は 0(その項目を持たない企業とみなす)、片方だけ欠損なら計算しない
- 支払利息・研究開発費の欠損は 0(論文の国際サンプルと同じ扱い)
- 投資:総資産成長率または自己資本成長率。論文に合わせ、1期前の成長率(t−2 → t−1)を使う。
  自己資本成長率は、両期で同じ項目(株主資本合計どうしなど)がそろうものだけで計算する。

In [ ]:
acct["TA"] = acct["TOTALBS"].where(acct["TOTALBS"] > 0)
be_raw = acct["CAPSTTL"].fillna(acct["EQTY"]).fillna(acct["NTASTTL"])
acct["BE"] = be_raw.where(be_raw > 0)
acct["BE_src"] = np.select([acct["CAPSTTL"].notna(), acct["EQTY"].notna(), acct["NTASTTL"].notna()],
                           ["株主資本合計", "資本合計", "純資産合計"], default="なし")

# 前期(同じ会計基準)の値を付ける
PREV_COLS = ["TA", "BE", "CAPSTTL", "EQTY", "NTASTTL", "ACRC", "INVENT", "ACPY"]
prev = acct[["NRI_CODE", "CTYPE", "fy_idx"] + PREV_COLS].copy()
prev["fy_idx"] += 12
prev["prev_exists"] = True
acct = acct.merge(prev.rename(columns={c: c + "_p" for c in PREV_COLS}),
                  on=["NRI_CODE", "CTYPE", "fy_idx"], how="left")
acct["prev_exists"] = acct["prev_exists"].eq(True)


def delta_item(cur, prv, exists):
    d = cur - prv
    d = d.mask(cur.isna() & prv.isna(), 0.0)  # 両期とも欠損 → その項目なし
    return d.where(exists)


acct["accrual"] = (delta_item(acct["ACRC"], acct["ACRC_p"], acct["prev_exists"])
                   + delta_item(acct["INVENT"], acct["INVENT_p"], acct["prev_exists"])
                   - delta_item(acct["ACPY"], acct["ACPY_p"], acct["prev_exists"]))
int_exp = acct["CHAGE"].fillna(0.0)
rd = acct["RD"].fillna(0.0)

numer = {
    "ROA": acct["OPINCM"],
    "ROE": acct["NETINCM"],
    "GPA": acct["GPLS"],
    "OP": acct["OPINCM"] - int_exp,
    "CBOP": acct["OPINCM"] + rd - acct["accrual"],
    "CP": acct["OPINCM"] - int_exp - acct["accrual"],
}
for m in MEASURES:
    denom = acct["TA_p"] if SCALE[m] == "asset" else acct["BE_p"]
    acct[m] = numer[m] / denom.where(denom > 0)

# 成長率(当期):総資産、自己資本(同じ項目どうし)
acct["g_TA"] = acct["TA"] / acct["TA_p"] - 1
g_be = pd.Series(np.nan, index=acct.index)
for col in ["CAPSTTL", "EQTY", "NTASTTL"]:
    ok = (acct[col] > 0) & (acct[col + "_p"] > 0) & g_be.isna()
    g_be[ok] = acct.loc[ok, col] / acct.loc[ok, col + "_p"] - 1
acct["g_BE"] = g_be

# 1期前(同じ会計基準)の収益性・成長率 と 1期後の収益性
LAG_COLS = MEASURES + ["g_TA", "g_BE"]
lag = acct[["NRI_CODE", "CTYPE", "fy_idx"] + LAG_COLS].copy()
lag["fy_idx"] += 12
acct = acct.merge(lag.rename(columns={c: c + "_lag" for c in LAG_COLS}),
                  on=["NRI_CODE", "CTYPE", "fy_idx"], how="left")
lead = acct[["NRI_CODE", "CTYPE", "fy_idx", "avail_date"] + MEASURES].copy()
lead["fy_idx"] -= 12
acct = acct.merge(lead.rename(columns={c: c + "_lead" for c in MEASURES + ["avail_date"]}),
                  on=["NRI_CODE", "CTYPE", "fy_idx"], how="left")
for m in MEASURES:
    acct["d" + m] = acct[m] - acct[m + "_lag"]

# 組成年への割り当て:入手可能日が組成月末までなら、その年の組成で使える
acct = acct[acct["avail_date"].notna()].copy()
av = acct["avail_date"]
acct["form_year"] = (av.dt.year + (av.dt.month > FORMATION_MONTH).astype(int)).astype(int)
form_midx = acct["form_year"] * 12 + FORMATION_MONTH - 1
acct["stale_ok"] = (form_midx - acct["fy_idx"]) <= MAX_STALENESS_MONTHS
acct["prio"] = acct["CTYPE"].map({c: i for i, c in enumerate(CTYPE_PRIORITY)}).fillna(99)

print("指標ごとの計算できた件数(会計基準別)")
display(acct.groupby(acct["CTYPE"].map(CTYPE_NAME).fillna(acct["CTYPE"]))[MEASURES].count().T)
print("指標の分布(全期間)")
display(acct[MEASURES].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T)

## 3. 組成年への割り当て・会計基準の選択・市場データとの結合

**会計基準の選択(指標ごと)**:各企業・各組成年について、使える決算のうち最新の決算期を選び、
そのうえで必要な変数(収益性・その変化・投資・自己資本)が **すべて同じ会計基準で計算できる** 行の中から、
`CTYPE_PRIORITY` の順で1つ選びます。

**市場データ**:組成日(6月末)時点で有効な証券コード・業種・TOPIX 構成を引き、
6月末の時価総額(規模)と決算期末の時価総額(B/M)を付けます。

**ユニバース**:業種が判明していて、電気・ガス・金融を除く銘柄。
マイクロキャップ = 6月末時価総額が TOPIX 構成銘柄の 20 パーセンタイル未満。

In [ ]:
# マスタの整形
code_tbl = clean_idsqe(raw["証券コード対応表"])
code_tbl["NRI_CODE"] = norm_sec(code_tbl["NRI_CODE"])
code_tbl["sec"] = norm_sec(code_tbl["ITEM_DATA"])
code_tbl["FROM_DATE"] = code_tbl["FROM_DATE"].fillna(0)
code_tbl["TO_DATE"] = code_tbl["TO_DATE"].fillna(99999999)

ind_tbl = clean_idsqe(raw["業種"])
ind_tbl["NRI_CODE"] = norm_sec(ind_tbl["NRI_CODE"])
ind_tbl["ind33"] = norm_code(ind_tbl["ITEM_DATA"]).str.zfill(4)
ind_tbl = ind_tbl[["NRI_CODE", "FROM_DATE", "TO_DATE", "ind33"]].drop_duplicates()
ind_tbl["FROM_DATE"] = ind_tbl["FROM_DATE"].fillna(0)
ind_tbl["TO_DATE"] = ind_tbl["TO_DATE"].fillna(99999999)
print(f"業種テーブル:重複除去後 {len(ind_tbl):,} 行(元 {len(raw['業種']):,} 行)")
unknown = sorted(set(ind_tbl["ind33"]) - set(IND33_NAME))
if unknown:
    print(f"[注意] 標準コード表にない業種コード:{unknown[:20]}")

topix_tbl = clean_idsqe(raw["TOPIX構成銘柄"])
topix_tbl["NRI_CODE"] = norm_sec(topix_tbl["NRI_CODE"])
topix_tbl = topix_tbl[["NRI_CODE", "FROM_DATE", "TO_DATE"]].drop_duplicates()
topix_tbl["FROM_DATE"] = topix_tbl["FROM_DATE"].fillna(0)
topix_tbl["TO_DATE"] = topix_tbl["TO_DATE"].fillna(99999999)
topix_tbl["topix"] = True

mcap = raw["時価総額"].copy()
mcap["sec"] = norm_sec(mcap["security_code"])
mcap["midx"] = to_midx(mcap["month"]) if "month" in mcap else to_midx(mcap["date"])
mcap["me"] = pd.to_numeric(mcap["market_cap"], errors="coerce")
mcap = mcap.loc[mcap["me"] > 0, ["sec", "midx", "me"]]
n_dup = mcap.duplicated(["sec", "midx"]).sum()
if n_dup:
    print(f"[注意] 時価総額:銘柄・月の重複 {n_dup} 行 → 最初の行を残します")
    mcap = mcap.drop_duplicates(["sec", "midx"])

ret = raw["月次リターン"].copy()
ret["sec"] = norm_sec(ret["security_code"])
ret["midx"] = to_midx(ret["month"])
ret["ret"] = pd.to_numeric(ret["ret"], errors="coerce")
ret = ret.loc[ret["ret"].notna(), ["sec", "midx", "ret"]]
n_dup = ret.duplicated(["sec", "midx"]).sum()
if n_dup:
    print(f"[注意] リターン:銘柄・月の重複 {n_dup} 行 → 最初の行を残します")
    ret = ret.drop_duplicates(["sec", "midx"])
print(f"リターンの外れ値チェック:-100%未満 {(ret['ret'] < -1).sum()} 件、+500%超 {(ret['ret'] > 5).sum()} 件")


def select_records(req_cols):
    """各企業・組成年について、req_cols がすべて計算できる行のうち、最新の決算期 → 会計基準の優先順で1行選ぶ"""
    d = acct[acct["stale_ok"] & acct["form_year"].between(START_YEAR - 1, END_YEAR)]
    d = d.dropna(subset=req_cols)
    d = d.sort_values(["NRI_CODE", "form_year", "fy_idx", "prio"], ascending=[True, True, False, True])
    return d.drop_duplicates(["NRI_CODE", "form_year"], keep="first").copy()


def attach_market(d):
    """組成日時点の証券コード・業種・TOPIX 構成、時価総額を付け、ユニバースで絞る"""
    d = d.copy()
    d["F_int"] = d["form_year"].map(formation_int)
    d = d.merge(asof_lookup(d, code_tbl, ["sec"], "F_int"), on=["NRI_CODE", "F_int"], how="left")
    d = d.merge(asof_lookup(d, ind_tbl, ["ind33"], "F_int"), on=["NRI_CODE", "F_int"], how="left")
    d = d.merge(asof_lookup(d, topix_tbl, ["topix"], "F_int"), on=["NRI_CODE", "F_int"], how="left")
    d["topix"] = d["topix"].eq(True)
    d["ind33_name"] = d["ind33"].map(IND33_NAME)
    d["ind17"] = d["ind33"].map(IND33_TO_17)
    d["midx_form"] = d["form_year"] * 12 + FORMATION_MONTH - 1
    d = d.merge(mcap.rename(columns={"midx": "midx_form", "me": "ME_form"}), on=["sec", "midx_form"], how="left")
    d = d.merge(mcap.rename(columns={"midx": "fy_idx", "me": "ME_fye"}), on=["sec", "fy_idx"], how="left")
    d = d[d["sec"].notna() & (d["ME_form"] > 0) & d["ind17"].notna() & ~d["ind33"].isin(EXCLUDE_IND33)].copy()
    d["lnME"] = np.log(d["ME_form"])
    bm = d["BE"] * ACCOUNTING_UNIT / d["ME_fye"]
    d["lnBM"] = np.log(bm.where(bm > 0))
    # マイクロキャップ判定:TOPIX 構成銘柄(足りなければ全銘柄)の 20 パーセンタイル
    bp = {}
    for y, g in d.groupby("form_year"):
        ref = g.loc[g["topix"], "ME_form"] if g["topix"].sum() >= MIN_OBS_CS else g["ME_form"]
        bp[y] = ref.quantile(MICRO_PCT)
    d["micro"] = d["ME_form"] < d["form_year"].map(bp)
    return d.drop_duplicates(["NRI_CODE", "form_year"])


def build_measure_panel(m):
    """指標 m のパネル(企業×組成年)。Y・ΔY・投資・コントロール変数をすべて同じ会計基準で持つ"""
    inv_col = "g_TA_lag" if SCALE[m] == "asset" else "g_BE_lag"
    d = select_records([m, "d" + m, inv_col, "BE"])
    d = d.rename(columns={m: "Y", "d" + m: "dY", m + "_lag": "Y_lag", m + "_lead": "Y_lead",
                          "avail_date_lead": "avail_lead", inv_col: "inv"})
    keep = ["NRI_CODE", "form_year", "CTYPE", "fy_idx", "fdate", "avail_date", "BE", "BE_src",
            "Y", "dY", "Y_lag", "Y_lead", "avail_lead", "inv"]
    d = attach_market(d[keep])
    d = d.dropna(subset=["lnBM"])
    d = winsorize_by(d, ["Y", "dY", "Y_lag", "inv", "lnBM"], "form_year")
    d["measure"] = m
    return d


panels = {}
for m in MEASURES:
    panels[m] = build_measure_panel(m)
    p = panels[m]
    print(f"{m}: {len(p):,} 企業年、組成年 {p['form_year'].min()}〜{p['form_year'].max()}、"
          f"会計基準 {p['CTYPE'].map(CTYPE_NAME).value_counts().to_dict()}")

print("組成年ごとの銘柄数(CBOP パネル)")
display(panels["CBOP"].groupby("form_year").agg(銘柄数=("NRI_CODE", "size"),
                                                マイクロ以外=("micro", lambda x: (~x).sum()),
                                                TOPIX=("topix", "sum")).T)

## 4. 第1段階:期待収益性の推定

論文の式(5)と同じく、毎年のクロスセクション回帰で翌期の収益性を予測します。

$$Y_{t+1} = \alpha + \beta_1 Y_t + \beta_2 \Delta Y_t + \gamma_1 \ln(B/M)_t + \gamma_2 \ln(ME)_t + \gamma_3 Inv_t + \varepsilon$$

- 推定はマイクロキャップを除く銘柄で行い、当てはめは全銘柄に行います(論文と同じ)。
- 被説明変数 $Y_{t+1}$ も毎年 winsorize します(日本データでは分母が小さい外れ値が多いため)。

期待収益性は2通り作ります。どちらも組成年 y に使う係数は、組成日までに実績 $Y_{t+1}$ が判明した
前年のコホート(y−1 年組成の企業)で推定したものです。

| 名前 | 作り方 | 情報の鮮度 |
|---|---|---|
| `Yhat_inv`(運用可能版) | 前年コホートで推定した係数を、**今年**の $X_t$ に当てはめる | 実績 Y と同じ時点の情報。先読みなし |
| `Yhat_pap`(論文版) | 前年コホートで推定した係数を、**前年コホート自身**の $X_{t-1}$ に当てはめる(サンプル内の当てはめ値) | 実績 Y より1年古い情報。論文の手順 |

In [ ]:
XCOLS = ["Y", "dY", "lnBM", "lnME", "inv"]


def first_stage(p):
    """期待収益性(Yhat_inv・Yhat_pap・Yhat_in)を付けたパネルと、年ごとの係数を返す"""
    p = p.copy()
    coef_rows, insample = {}, []
    for y in sorted(p["form_year"].unique()):
        F = formation_date(y)
        tr = p[(p["form_year"] == y - 1) & ~p["micro"] & (p["avail_lead"] <= F)].dropna(subset=XCOLS + ["Y_lead"])
        if len(tr) < MIN_OBS_CS:
            continue
        lo, hi = tr["Y_lead"].quantile(WINSOR)
        b, adj, n = ols_cs(tr["Y_lead"].clip(lo, hi).to_numpy(float), tr[XCOLS].to_numpy(float))
        coef_rows[y] = np.r_[b, adj, n]
        coh = p[p["form_year"] == y - 1].dropna(subset=XCOLS)
        insample.append(pd.DataFrame({"NRI_CODE": coh["NRI_CODE"].values, "cohort": y - 1,
                                      "fit": b[0] + coh[XCOLS].to_numpy(float) @ b[1:]}))
    if not coef_rows:
        raise ValueError("第1段階の回帰を推定できる年がありません(サンプルや入手可能日を確認してください)")
    coef = pd.DataFrame.from_dict(coef_rows, orient="index", columns=["const"] + XCOLS + ["adjR2", "N"]).sort_index()
    coef_avg = coef[["const"] + XCOLS].rolling(FIRST_STAGE_YEARS, min_periods=1).mean()
    p["Yhat_inv"] = np.nan
    for y, b in coef_avg.iterrows():
        mask = p["form_year"] == y
        p.loc[mask, "Yhat_inv"] = b["const"] + p.loc[mask, XCOLS].to_numpy(float) @ b[XCOLS].to_numpy(float)
    ins = pd.concat(insample, ignore_index=True)
    # 論文版:前年コホートの当てはめ値を、今年の組成で使う
    p = p.merge(ins.assign(form_year=ins["cohort"] + 1)[["NRI_CODE", "form_year", "fit"]]
                .rename(columns={"fit": "Yhat_pap"}), on=["NRI_CODE", "form_year"], how="left")
    # サンプル内の当てはめ値(予測精度の比較用、コホートそのものに付ける)
    p = p.merge(ins.rename(columns={"cohort": "form_year", "fit": "Yhat_in"}), on=["NRI_CODE", "form_year"], how="left")
    return p, coef


first_stage_coef = {}
for m in MEASURES:
    panels[m], first_stage_coef[m] = first_stage(panels[m])

print("第1段階の係数(年次回帰の時系列平均)")
display(pd.DataFrame({m: c.mean() for m, c in first_stage_coef.items()}).T)

### 4-1. 将来の収益性の予測回帰(論文 Table 1 に相当)

共通サンプル(マイクロキャップを除く)で3つの式を比べます。

- (A) $Y_t$ + コントロール
- (B) $Y_t, Y_{t-1}$ + コントロール
- (C) $Y_t, \Delta Y_t$ + コントロール

偏 R² = (R²(C) − R²(A)) / (1 − R²(A)) で、過去の成長を加えた説明力の上乗せを見ます。

In [ ]:
def profitability_regressions(p):
    specs = {"(A) Y": ["Y"], "(B) Y, Y_lag": ["Y", "Y_lag"], "(C) Y, ΔY": ["Y", "dY"]}
    ctrl = ["lnBM", "lnME", "inv"]
    s = p[~p["micro"]].dropna(subset=["Y", "Y_lag", "dY", "Y_lead"] + ctrl)
    rows = {k: {} for k in specs}
    for y, g in s.groupby("form_year"):
        if len(g) < MIN_OBS_CS:
            continue
        lo, hi = g["Y_lead"].quantile(WINSOR)
        yv = g["Y_lead"].clip(lo, hi).to_numpy(float)
        for k, xs in specs.items():
            b, adj, n = ols_cs(yv, g[xs + ctrl].to_numpy(float))
            rows[k][y] = np.r_[b, adj, n]
    out = {}
    for k, xs in specs.items():
        df = pd.DataFrame.from_dict(rows[k], orient="index", columns=["const"] + xs + ctrl + ["adjR2", "N"])
        col = {}
        for v in xs + ctrl:
            mu, t, _ = mean_t(df[v], NW_LAGS_ANNUAL)
            col[(v, "係数")] = mu
            col[(v, "t値")] = t
        col[("平均adjR2", "")] = df["adjR2"].mean()
        col[("平均N", "")] = df["N"].mean()
        out[k] = pd.Series(col)
    out = pd.DataFrame(out)
    r2a, r2c = out.loc[("平均adjR2", ""), "(A) Y"], out.loc[("平均adjR2", ""), "(C) Y, ΔY"]
    out.loc[("偏R2(C vs A)", ""), "(C) Y, ΔY"] = (r2c - r2a) / (1 - r2a)
    order = [(v, s) for v in ["Y", "Y_lag", "dY"] + ctrl for s in ["係数", "t値"]]
    order += [("平均adjR2", ""), ("平均N", ""), ("偏R2(C vs A)", "")]
    return out.reindex(pd.MultiIndex.from_tuples(order))


table1 = {}
for m in MEASURES:
    table1[m] = profitability_regressions(panels[m])
    print(f"■ {m}")
    display(table1[m])

### 4-2. 予測精度の比較

同じ予測対象($Y_{t+1}$)について、3つの予測値の精度を比べます(マイクロキャップ除く・共通サンプル)。

- 実績(ランダムウォーク):$Y_t$ をそのまま予測値とする
- 期待・運用可能版(サンプル外):`Yhat_inv`
- 期待・サンプル内:そのコホート自身で推定した回帰の当てはめ値(先読みあり。上限の目安)

指標は毎年計算して平均します。ファクターは順位でソートするので、**順位相関** が最も重要です。

会社予想・アナリスト予想は、データを取得したらこの表に列を追加します。

In [ ]:
def forecast_accuracy(p):
    preds = {"実績(ランダムウォーク)": "Y", "期待・運用可能(サンプル外)": "Yhat_inv", "期待・サンプル内": "Yhat_in"}
    s = p[~p["micro"]].dropna(subset=["Y_lead"] + list(preds.values()))
    per_year = []
    for y, g in s.groupby("form_year"):
        if len(g) < MIN_OBS_CS:
            continue
        lo, hi = g["Y_lead"].quantile(WINSOR)
        tgt = g["Y_lead"].clip(lo, hi)
        r_tgt = tgt.rank()
        for name, col in preds.items():
            e = tgt - g[col]
            per_year.append({"year": y, "予測": name,
                             "順位相関": np.corrcoef(r_tgt, g[col].rank())[0, 1],
                             "MAE": e.abs().mean(),
                             "R2(サンプル外)": 1 - (e ** 2).sum() / ((tgt - tgt.mean()) ** 2).sum()})
    py = pd.DataFrame(per_year)
    summary = py.groupby("予測")[["順位相関", "MAE", "R2(サンプル外)"]].mean()
    wide = py.pivot(index="year", columns="予測", values="順位相関")
    diff = wide["期待・運用可能(サンプル外)"] - wide["実績(ランダムウォーク)"]
    mu, t, _ = mean_t(diff, NW_LAGS_ANNUAL)
    summary.loc["期待・運用可能(サンプル外)", "順位相関の差(対実績)"] = mu
    summary.loc["期待・運用可能(サンプル外)", "差のt値"] = t
    return summary.loc[list(preds)]


accuracy = {}
for m in MEASURES:
    accuracy[m] = forecast_accuracy(panels[m])
    print(f"■ {m}")
    display(accuracy[m])

## 5. 第2段階:Fama-MacBeth 回帰

毎月、月次リターンを収益性とコントロール変数に回帰し、係数の時系列平均と Newey-West の t 値を出します
(係数は ×100)。保有期間は組成年 y の7月〜翌年6月です。

- コントロール:ln(B/M)、ln(ME)、投資、前月リターン R(1,1)、t−12〜t−2 の11か月リターン R(2,12)
- ユニバース:全銘柄 / マイクロキャップ除く。加重:等加重(OLS) / 時価総額加重(前月末時価総額で WLS)
- すべての式で同じサンプル(全変数がそろう銘柄・月)を使います。

| 式 | 収益性の変数 |
|---|---|
| (1) | $Y_t$ |
| (2) | $Y_t$, $\Delta Y_t$ |
| (3) | $Y_t$, $\hat{Y}$(運用可能) |
| (4) | $\hat{Y}$(運用可能) |
| (5) | $Y_t$, $\hat{Y}$(論文) |
| (6) | $\hat{Y}$(論文) |

「IQR リターン差」= 毎月の係数 × その月の変数の四分位範囲 の平均(%/月)。収益性が 25→75 パーセンタイルに
変わったときの期待リターンの差で、(4)・(6) と (1) の差の t 値も出します。

In [ ]:
# 月次パネル(リターン、前月末時価総額、モメンタム)
secs = set()
for p in panels.values():
    secs |= set(p["sec"])
r_wide = ret[ret["sec"].isin(secs)].pivot(index="midx", columns="sec", values="ret")
r_wide = r_wide.reindex(range(r_wide.index.min(), r_wide.index.max() + 1))
me_wide = mcap[mcap["sec"].isin(secs)].pivot(index="midx", columns="sec", values="me")
me_wide = me_wide.reindex(index=r_wide.index, columns=r_wide.columns)
r11 = r_wide.shift(1)
r212 = np.expm1(np.log1p(r_wide.clip(lower=-0.99)).shift(2).rolling(11, min_periods=11).sum())
me_lag = me_wide.shift(1)

T_, N_ = r_wide.shape
mlong = pd.DataFrame({
    "midx": np.repeat(r_wide.index.to_numpy(), N_),
    "sec": np.tile(r_wide.columns.to_numpy(), T_),
    "ret": r_wide.to_numpy().ravel(),
    "R11": r11.to_numpy().ravel(),
    "R212": r212.to_numpy().ravel(),
    "me_lag": me_lag.to_numpy().ravel(),
})
mlong = mlong[mlong["ret"].notna()].copy()
mlong["form_year"] = (mlong["midx"] - FORMATION_MONTH) // 12
mlong = mlong[mlong["form_year"].between(START_YEAR, END_YEAR)]
print(f"月次パネル:{len(mlong):,} 行、{mlong['midx'].nunique()} か月")
del r_wide, me_wide, r11, r212, me_lag

In [ ]:
CONTROLS = ["lnBM", "lnME", "inv", "R11", "R212"]
SPECS = {
    "(1) Y": ["Y"],
    "(2) Y+ΔY": ["Y", "dY"],
    "(3) Y+Ŷ運用": ["Y", "Yhat_inv"],
    "(4) Ŷ運用": ["Yhat_inv"],
    "(5) Y+Ŷ論文": ["Y", "Yhat_pap"],
    "(6) Ŷ論文": ["Yhat_pap"],
}
UNIVERSES = {"全銘柄": lambda d: d, "マイクロ除く": lambda d: d[~d["micro"]]}
WEIGHTS = {"等加重": None, "時価総額加重": "me_lag"}


def fm_data(m):
    p = panels[m]
    cols = ["NRI_CODE", "sec", "form_year", "micro", "ind17", "Y", "dY", "Yhat_inv", "Yhat_pap", "lnBM", "lnME", "inv"]
    d = p[cols].merge(mlong, on=["sec", "form_year"], how="inner")
    return d.dropna(subset=["ret", "Y", "dY", "Yhat_inv", "Yhat_pap", "me_lag"] + CONTROLS)


fm_detail, fm_summary_rows = {}, []
for m in MEASURES:
    d_all = fm_data(m)
    for uname, ufun in UNIVERSES.items():
        d = ufun(d_all)
        for wname, wcol in WEIGHTS.items():
            res = {s: fama_macbeth(d, "ret", xs + CONTROLS, wcol) for s, xs in SPECS.items()}
            fm_detail[(m, uname, wname)] = res
            row = {"指標": m, "ユニバース": uname, "加重": wname}
            for label, spec, var in [("Y (1)", "(1) Y", "Y"), ("ΔY (2)", "(2) Y+ΔY", "dY"),
                                     ("Ŷ運用 (4)", "(4) Ŷ運用", "Yhat_inv"), ("Ŷ論文 (6)", "(6) Ŷ論文", "Yhat_pap")]:
                mu, t, _ = mean_t(res[spec]["coef"][var])
                row[f"{label} 係数"] = mu * 100
                row[f"{label} t"] = t
            iqr_y = res["(1) Y"]["coef"]["Y"] * res["(1) Y"]["iqr"]["Y"]
            row["IQR差 Y (1)"] = iqr_y.mean() * 100
            for label, spec, var in [("Ŷ運用 (4)", "(4) Ŷ運用", "Yhat_inv"), ("Ŷ論文 (6)", "(6) Ŷ論文", "Yhat_pap")]:
                iqr_h = res[spec]["coef"][var] * res[spec]["iqr"][var]
                row[f"IQR差 {label}"] = iqr_h.mean() * 100
                _, t, _ = mean_t(iqr_h - iqr_y)
                row[f"IQR差 {label} - Y の t"] = t
            row["平均N"] = res["(1) Y"]["n"].mean()
            fm_summary_rows.append(row)
    print(f"{m} 完了")

fm_summary = pd.DataFrame(fm_summary_rows)
display(fm_summary)

### 5-1. 式ごとの詳細(例:CBOP・全銘柄・等加重)

`show_fm(指標, ユニバース, 加重)` でほかの組み合わせも表示できます。

In [ ]:
def show_fm(m, uname="全銘柄", wname="等加重"):
    res = fm_detail[(m, uname, wname)]
    tbl = pd.concat({s: summarize_fm(r) for s, r in res.items()}, axis=1)
    order = ["Y", "dY", "Yhat_inv", "Yhat_pap"] + CONTROLS + ["const", "平均N", "平均adjR2"]
    return tbl.reindex(order)


display(show_fm("CBOP"))
display(show_fm("CP"))

## 6. ファクターリターン(2×3 ソート)

Fama-French の RMW と同じ作り方です。毎年6月末に

1. 規模:TOPIX 構成銘柄の時価総額の中央値で 小型(S)/大型(B)
2. シグナル:TOPIX 構成銘柄の 30・70 パーセンタイルで 低(L)/中(M)/高(H)

に分け、7月〜翌6月の時価総額加重リターン(前月末時価総額で加重)を計算します。

ファクター = (S/H + B/H)/2 − (S/L + B/L)/2。小型だけ(S/H − S/L)・大型だけ(B/H − B/L)のスプレッドも出します。

**シグナル**:実績 Y、期待 Ŷ(運用可能・論文)、業種中立版(17業種内のパーセンタイル順位でソート)

**アルファとスパニング回帰**:Ken French サイトの日本の5ファクター(`data/Japan_5_Factors.csv`)を使います。

| 回帰 | 被説明変数 | 説明変数 | 見ていること |
|---|---|---|---|
| FF5 アルファ | 自作ファクター | Mkt-RF, SMB, HML, RMW, CMA | 既存の RMW を含む5ファクターで説明できない部分があるか |
| FF4 アルファ | 自作ファクター | Mkt-RF, SMB, HML, CMA | RMW を除いたときのアルファ(収益性プレミアムの大きさ) |
| RMW の説明 | RMW | Mkt-RF, SMB, HML, CMA + 自作ファクター | 自作ファクターが RMW を包含するか(アルファが消えれば包含) |
| 期待 vs 実績 | 期待ファクター | FF5 + 実績ファクター(とその逆) | 期待の情報が実績を超えるか |

注意:French の国際ファクターは **米ドル建て** で、RF は米国 T-bill です。ロングショートのファクター
(SMB・HML・RMW・CMA)は通貨の影響をほぼ打ち消しますが、Mkt-RF には為替の変動が含まれます。
自作ファクターは円建てのロングショートなので、無リスク金利は差し引いていません。

In [ ]:
def sort_2x3(df, signal, bps=(0.3, 0.7)):
    d = df.dropna(subset=[signal, "ME_form"]).copy()
    rows = []
    for y, g in d.groupby("form_year"):
        r = g[g["topix"]] if g["topix"].sum() >= MIN_OBS_CS else g
        rows.append((y, r["ME_form"].median(), r[signal].quantile(bps[0]), r[signal].quantile(bps[1])))
    bp = pd.DataFrame(rows, columns=["form_year", "size_bp", "lo", "hi"])
    d = d.merge(bp, on="form_year")
    size = np.where(d["ME_form"] <= d["size_bp"], "S", "B")
    sig = np.where(d[signal] <= d["lo"], "L", np.where(d[signal] >= d["hi"], "H", "M"))
    d["grp"] = pd.Series(size, index=d.index) + "/" + pd.Series(sig, index=d.index)
    return d


def portfolio_returns(hold, group_col):
    """組成年ごとの保有銘柄から、月次の時価総額加重リターンと銘柄数を計算"""
    d = hold[["sec", "form_year", group_col]].merge(mlong[["sec", "form_year", "midx", "ret", "me_lag"]],
                                                     on=["sec", "form_year"])
    d = d[(d["me_lag"] > 0) & d["ret"].notna()]
    d["wr"] = d["ret"] * d["me_lag"]
    g = d.groupby(["midx", group_col])
    rets = (g["wr"].sum() / g["me_lag"].sum()).unstack(group_col)
    counts = g.size().unstack(group_col)
    return rets, counts


def make_factor(df, signal, high_minus_low=True):
    h = sort_2x3(df, signal)
    rets, counts = portfolio_returns(h, "grp")
    hi, lo = ("H", "L") if high_minus_low else ("L", "H")
    f = (rets[f"S/{hi}"] + rets[f"B/{hi}"]) / 2 - (rets[f"S/{lo}"] + rets[f"B/{lo}"]) / 2
    small = rets[f"S/{hi}"] - rets[f"S/{lo}"]
    big = rets[f"B/{hi}"] - rets[f"B/{lo}"]
    smb = rets[[c for c in rets if c.startswith("S/")]].mean(axis=1) - rets[[c for c in rets if c.startswith("B/")]].mean(axis=1)
    return {"factor": f, "small": small, "big": big, "smb": smb, "rets": rets, "counts": counts}


def load_french_factors(path):
    """Ken French サイトの CSV(月次部分)を読み込む。% → 小数、インデックスは通し月番号"""
    lines = Path(path).read_text(encoding="utf-8-sig", errors="ignore").splitlines()
    start = next(i for i, l in enumerate(lines) if "Mkt-RF" in l)
    cols = [c.strip() for c in lines[start].split(",")][1:]
    rows = []
    for l in lines[start + 1:]:
        parts = [x.strip() for x in l.split(",")]
        if len(parts[0]) != 6 or not parts[0].isdigit():
            break  # 月次部分の終わり(年次部分の手前)
        rows.append(parts[: len(cols) + 1])
    df = pd.DataFrame(rows, columns=["yyyymm"] + cols)
    vals = df[cols].astype(float).mask(lambda x: x <= -99.99) / 100
    vals.index = to_midx(df["yyyymm"].astype(int)).to_numpy()
    return vals


if FF5_PATH.exists():
    ff5 = load_french_factors(FF5_PATH)
    print(f"FF5:{midx_to_ts([ff5.index.min()])[0]:%Y-%m}〜{midx_to_ts([ff5.index.max()])[0]:%Y-%m}、列 {list(ff5.columns)}")
    display(pd.DataFrame({c: dict(zip(["平均(%/月)", "t値", "月数"], mean_t(ff5[c])))
                          for c in ff5.columns if c != "RF"}).T
            .assign(**{"平均(%/月)": lambda x: x["平均(%/月)"] * 100}))
else:
    ff5 = None
    print(f"[注意] {FF5_PATH} が見つかりません。アルファとスパニング回帰は計算しません。")

FF5_COLS = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"]
FF4_COLS = ["Mkt-RF", "SMB", "HML", "CMA"]


def regress(f, X):
    """f を X(DataFrame)に回帰:アルファ(%/月)・t値と各係数・t値"""
    d = pd.concat([f.rename("f"), X], axis=1).dropna()
    b, tb, T = ols_nw(d["f"].values, d[X.columns].values, NW_LAGS_MONTHLY)
    out = {"α(%/月)": b[0] * 100, "αのt値": tb[0], "月数": T}
    for i, c in enumerate(X.columns, start=1):
        out[f"{c}係数"] = b[i]
        out[f"{c}のt値"] = tb[i]
    return out


def factor_row(f):
    f = f.dropna()
    mu, t, T = mean_t(f)
    row = {"平均(%/月)": mu * 100, "t値": t, "年率SR": f.mean() / f.std() * np.sqrt(12), "月数": T}
    if ff5 is not None:
        r5 = regress(f, ff5[FF5_COLS])
        r4 = regress(f, ff5[FF4_COLS])
        row.update({"FF5 α": r5["α(%/月)"], "FF5 αのt値": r5["αのt値"],
                    "RMW係数": r5["RMW係数"], "RMWのt値": r5["RMWのt値"],
                    "FF4 α": r4["α(%/月)"], "FF4 αのt値": r4["αのt値"]})
    return row

In [ ]:
SIGNALS = {
    "実績Y": "Y",
    "期待Ŷ(運用可能)": "Yhat_inv",
    "期待Ŷ(論文)": "Yhat_pap",
    "業種中立 実績Y": "Y_ind",
    "業種中立 期待Ŷ(運用可能)": "Yhat_inv_ind",
}
factors, factor_tables, spanning_rows = {}, {}, []
for m in MEASURES:
    p = panels[m].dropna(subset=["Y", "Yhat_inv", "Yhat_pap"]).copy()
    p = p[p["form_year"].between(START_YEAR, END_YEAR)]
    p["Y_ind"] = p.groupby(["form_year", "ind17"])["Y"].rank(pct=True)
    p["Yhat_inv_ind"] = p.groupby(["form_year", "ind17"])["Yhat_inv"].rank(pct=True)
    rows = {}
    for sname, col in SIGNALS.items():
        fac = make_factor(p, col)
        factors[(m, sname)] = fac
        r = factor_row(fac["factor"])
        for leg in ["small", "big"]:
            mu, t, _ = mean_t(fac[leg])
            r[f"{'小型' if leg == 'small' else '大型'} 平均"] = mu * 100
            r[f"{'小型' if leg == 'small' else '大型'} t値"] = t
        rows[sname] = r
    tbl = pd.DataFrame(rows).T
    # 期待 − 実績 の差
    for sname in ["期待Ŷ(運用可能)", "期待Ŷ(論文)", "業種中立 実績Y", "業種中立 期待Ŷ(運用可能)"]:
        mu, t, _ = mean_t(factors[(m, sname)]["factor"] - factors[(m, "実績Y")]["factor"])
        tbl.loc[sname, "対実績Y 差(%/月)"] = mu * 100
        tbl.loc[sname, "差のt値"] = t
    factor_tables[m] = tbl
    # スパニング回帰(French の FF5 を使用)
    if ff5 is not None:
        fy, fh = factors[(m, "実績Y")]["factor"], factors[(m, "期待Ŷ(運用可能)")]["factor"]
        tests = [
            # (被説明の名前, 被説明, 説明変数の名前, 説明変数)
            ("RMW", ff5["RMW"], "FF4 + 実績Y", pd.concat([ff5[FF4_COLS], fy.rename("自作")], axis=1)),
            ("RMW", ff5["RMW"], "FF4 + 期待Ŷ(運用可能)", pd.concat([ff5[FF4_COLS], fh.rename("自作")], axis=1)),
            ("期待Ŷ(運用可能)", fh, "FF5 + 実績Y", pd.concat([ff5[FF5_COLS], fy.rename("自作")], axis=1)),
            ("実績Y", fy, "FF5 + 期待Ŷ(運用可能)", pd.concat([ff5[FF5_COLS], fh.rename("自作")], axis=1)),
        ]
        for lhs_name, lhs, rhs_name, X in tests:
            r = regress(lhs, X)
            spanning_rows.append({"指標": m, "被説明": lhs_name, "説明変数": rhs_name,
                                  "α(%/月)": r["α(%/月)"], "αのt値": r["αのt値"],
                                  "追加した自作ファクターの係数": r["自作係数"], "係数のt値": r["自作のt値"],
                                  "月数": r["月数"]})
    print(f"■ {m}")
    display(tbl)

spanning = pd.DataFrame(spanning_rows)
if len(spanning):
    print("スパニング回帰(French の日本5ファクターを使用)")
    display(spanning)

In [ ]:
# 累積リターン(対数)の比較
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
for ax, m in zip(axes.ravel(), MEASURES):
    for sname in ["実績Y", "期待Ŷ(運用可能)", "業種中立 実績Y"]:
        f = factors[(m, sname)]["factor"].dropna()
        ax.plot(midx_to_ts(f.index.to_numpy()), np.log1p(f).cumsum(), label=sname)
    ax.set_title(m)
    ax.axhline(0, color="gray", lw=0.5)
axes[0, 0].legend(prop={"family": ["Yu Gothic", "Meiryo", "MS Gothic", "sans-serif"]})
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "factor_cumulative.png", dpi=120)
plt.show()

## 7. 業種別分析

### 7-1. 業種ごとの収益性の水準
毎年の業種中央値を時系列平均したもの。業種で水準が大きく違えば、全体でソートしたファクターは
業種への偏り(業種ベット)を含みます。

In [ ]:
ind_level = pd.DataFrame({m: panels[m].groupby(["form_year", "ind17"])["Y"].median().groupby("ind17").mean()
                          for m in MEASURES})
ind_level["平均銘柄数"] = panels["CBOP"].groupby(["form_year", "ind17"]).size().groupby("ind17").mean()
display(ind_level.sort_values("CBOP"))

### 7-2. 業種内スプレッド(規模調整あり)

規模の影響(小型株効果)が混ざらないように、業種ごとに次の手順で 2×3 に分けます。

1. 業種内の6月末時価総額の **中央値** で 小型(S)/大型(B)に分ける
2. 業種 × 規模 の中で、シグナルの 30・70 パーセンタイルで 低(L)/中(M)/高(H)に分ける
3. スプレッド = (S/H − S/L + B/H − B/L)/ 2(時価総額加重)

業種によっては銘柄が少ないため、全体のファクターと違い、ブレークポイントは
**業種内で計算**し、シグナルは **規模グループ内で分ける(従属ソート)** ことで、6つのセルに必ず銘柄が入るようにしています。
比較のため、規模調整なし(業種内で3分割のみ)の結果も出します。

- 6つのセルのうち最も少ないセルの平均銘柄数が 5 未満の業種は「少数」と表示します(結果が不安定)。
- 17業種で検定するので、t 値 2 程度の業種が1〜2個あっても偶然の可能性があります(多重検定)。

In [ ]:
def industry_spreads(p, signal, size_adjust=True):
    d = p.dropna(subset=[signal, "ME_form"]).copy()
    d = d[d["form_year"].between(START_YEAR, END_YEAR)]
    keys = ["form_year", "ind17"]
    if size_adjust:
        med = d.groupby(keys)["ME_form"].transform("median")
        d["size"] = np.where(d["ME_form"] <= med, "S", "B")
    else:
        d["size"] = "A"
    rk = d.groupby(keys + ["size"])[signal].rank(pct=True)
    d["terc"] = np.where(rk <= 0.3, "L", np.where(rk > 0.7, "H", "M"))
    d["grp"] = d["ind17"] + "|" + d["size"] + "|" + d["terc"]
    rets, counts = portfolio_returns(d, "grp")
    sizes = ["S", "B"] if size_adjust else ["A"]
    rows = {}
    for ind in sorted(d["ind17"].unique()):
        cells = [f"{ind}|{s}|{q}" for s in sizes for q in ["H", "L"]]
        if any(c not in rets for c in cells):
            continue
        legs = pd.concat([rets[f"{ind}|{s}|H"] - rets[f"{ind}|{s}|L"] for s in sizes], axis=1)
        hl = legs.mean(axis=1, skipna=False)
        mu, t, T = mean_t(hl)
        n_cell = min(counts[c].mean() for c in cells)
        row = {"平均(%/月)": mu * 100, "t値": t, "月数": T, "最少セルの平均銘柄数": n_cell,
               "注意": "少数" if n_cell < 5 else ""}
        if size_adjust:
            for s, name in [("S", "小型"), ("B", "大型")]:
                mu_s, t_s, _ = mean_t(rets[f"{ind}|{s}|H"] - rets[f"{ind}|{s}|L"])
                row[f"{name} 平均"] = mu_s * 100
                row[f"{name} t値"] = t_s
        rows[ind] = row
    return pd.DataFrame(rows).T


ind_spreads = {}
for m in MEASURES:
    for sname, col in [("実績Y", "Y"), ("期待Ŷ(運用可能)", "Yhat_inv")]:
        for adj in [True, False]:
            ind_spreads[(m, sname, "規模調整あり" if adj else "規模調整なし")] = industry_spreads(panels[m], col, adj)

for m in ["CBOP", "ROA"]:
    print(f"■ {m}:規模調整あり")
    compare = pd.concat({s: ind_spreads[(m, s, "規模調整あり")][["平均(%/月)", "t値", "小型 平均", "大型 平均"]]
                         for s in ["実績Y", "期待Ŷ(運用可能)"]}, axis=1)
    compare[("規模調整なし(実績Y)", "平均(%/月)")] = ind_spreads[(m, "実績Y", "規模調整なし")]["平均(%/月)"]
    compare[("規模調整なし(実績Y)", "t値")] = ind_spreads[(m, "実績Y", "規模調整なし")]["t値"]
    compare[("", "最少セルの平均銘柄数")] = ind_spreads[(m, "実績Y", "規模調整あり")]["最少セルの平均銘柄数"]
    display(compare)

## 8. まとめ:どの指標が優れているか、期待にするとどうなるか

各分析の主要な数字を、指標ごとに1つの表に集めます。

| 観点 | 使う数字 |
|---|---|
| 将来の収益性の予測 | 偏 R²(ΔY による説明力の上乗せ)、順位相関(実績・期待)と差の t 値 |
| リターンの説明(FM 回帰) | 全銘柄・等加重 と マイクロ除く・時価総額加重 の、Y と Ŷ の t 値、IQR リターン差とその差の t 値 |
| ファクター | 実績・期待ファクターの平均・t 値・シャープレシオ・FF5 アルファの t 値、期待 − 実績 の差の t 値 |

「期待の効果」は、期待 − 実績 の差の符号と t 値から「改善(有意)/改善/悪化/悪化(有意)」と判定します(|t| ≥ 1.96 で有意)。

In [ ]:
def judge(diff, t):
    if pd.isna(diff) or pd.isna(t):
        return ""
    if diff > 0:
        return "改善(有意)" if t >= 1.96 else "改善"
    return "悪化(有意)" if t <= -1.96 else "悪化"


summary_rows = []
for m in MEASURES:
    r = {"指標": m}
    # 予測
    t1 = table1[m]
    r[("予測", "偏R2(ΔY)")] = t1.loc[("偏R2(C vs A)", ""), "(C) Y, ΔY"]
    r[("予測", "ΔYのt値")] = t1.loc[("dY", "t値"), "(C) Y, ΔY"]
    acc = accuracy[m]
    r[("予測", "順位相関 実績")] = acc.loc["実績(ランダムウォーク)", "順位相関"]
    r[("予測", "順位相関 期待")] = acc.loc["期待・運用可能(サンプル外)", "順位相関"]
    r[("予測", "期待の効果")] = judge(acc.loc["期待・運用可能(サンプル外)", "順位相関の差(対実績)"],
                                 acc.loc["期待・運用可能(サンプル外)", "差のt値"])
    # FM 回帰
    for u, w, lab in [("全銘柄", "等加重", "FM全・等"), ("マイクロ除く", "時価総額加重", "FM大中・VW")]:
        s = fm_summary[(fm_summary["指標"] == m) & (fm_summary["ユニバース"] == u) & (fm_summary["加重"] == w)].iloc[0]
        r[(lab, "Y t値")] = s["Y (1) t"]
        r[(lab, "Ŷ t値")] = s["Ŷ運用 (4) t"]
        r[(lab, "IQR差 Y")] = s["IQR差 Y (1)"]
        r[(lab, "IQR差 Ŷ")] = s["IQR差 Ŷ運用 (4)"]
        r[(lab, "期待の効果")] = judge(s["IQR差 Ŷ運用 (4)"] - s["IQR差 Y (1)"], s["IQR差 Ŷ運用 (4) - Y の t"])
    # ファクター
    ft = factor_tables[m]
    for sname, lab in [("実績Y", "実績"), ("期待Ŷ(運用可能)", "期待")]:
        r[("ファクター", f"{lab} 平均")] = ft.loc[sname, "平均(%/月)"]
        r[("ファクター", f"{lab} t値")] = ft.loc[sname, "t値"]
        r[("ファクター", f"{lab} SR")] = ft.loc[sname, "年率SR"]
        if "FF5 αのt値" in ft:
            r[("ファクター", f"{lab} FF5α t値")] = ft.loc[sname, "FF5 αのt値"]
    r[("ファクター", "期待の効果")] = judge(ft.loc["期待Ŷ(運用可能)", "対実績Y 差(%/月)"],
                                    ft.loc["期待Ŷ(運用可能)", "差のt値"])
    r[("ファクター", "業種中立 実績 SR")] = ft.loc["業種中立 実績Y", "年率SR"]
    summary_rows.append(r)

measure_summary = pd.DataFrame(summary_rows).set_index("指標")
measure_summary.columns = pd.MultiIndex.from_tuples(measure_summary.columns)
display(measure_summary)

### 8-1. 指標の順位

実績の収益性について、観点ごとに指標を順位付けします(1 が最良)。
「平均順位」が小さいほど、全体として優れた指標です。

In [ ]:
rank_basis = pd.DataFrame({
    "予測:順位相関": measure_summary[("予測", "順位相関 実績")],
    "FM全・等:t値": measure_summary[("FM全・等", "Y t値")],
    "FM大中・VW:t値": measure_summary[("FM大中・VW", "Y t値")],
    "ファクター:SR": measure_summary[("ファクター", "実績 SR")],
})
if ("ファクター", "実績 FF5α t値") in measure_summary:
    rank_basis["ファクター:FF5α t値"] = measure_summary[("ファクター", "実績 FF5α t値")]
ranks = rank_basis.rank(ascending=False)
ranks["平均順位"] = ranks.mean(axis=1)
display(pd.concat({"値": rank_basis, "順位": ranks}, axis=1).sort_values(("順位", "平均順位")))

# 自動の要約文
best = ranks["平均順位"].idxmin()
print(f"・平均順位が最も良い指標:{best}")
for col, lab in [(("予測", "期待の効果"), "将来の収益性の予測"), (("FM全・等", "期待の効果"), "FM回帰(全銘柄・等加重)"),
                 (("FM大中・VW", "期待の効果"), "FM回帰(マイクロ除く・時価総額加重)"), (("ファクター", "期待の効果"), "ファクターリターン")]:
    vc = measure_summary[col].value_counts()
    print(f"・期待にした効果({lab}):" + "、".join(f"{k} {v}指標" for k, v in vc.items()))

## 9. 結果の保存

`outputs/` フォルダに CSV(Excel で開ける UTF-8 BOM 付き)で保存します。

In [ ]:
def save(df, name):
    df.to_csv(OUTPUT_DIR / f"{name}.csv", encoding="utf-8-sig")


for m in MEASURES:
    save(table1[m], f"1_profitability_regression_{m}")
    save(accuracy[m], f"2_forecast_accuracy_{m}")
    save(factor_tables[m], f"4_factor_{m}")
    for (mm, s, adj), tbl in ind_spreads.items():
        if mm == m:
            save(tbl, f"5_industry_spread_{m}_{s}_{adj}")
save(fm_summary, "3_fama_macbeth_summary")
for (m, u, w), res in fm_detail.items():
    save(show_fm(m, u, w), f"3_fama_macbeth_{m}_{u}_{w}")
save(spanning, "4_spanning")
save(ind_level, "5_industry_level")
save(pd.DataFrame({f"{m}|{s}": f["factor"] for (m, s), f in factors.items()}), "4_factor_returns_monthly")
save(measure_summary, "6_measure_summary")
save(pd.concat({"値": rank_basis, "順位": ranks}, axis=1), "6_measure_ranking")
print(f"保存先:{OUTPUT_DIR.resolve()}")